## 1. Setup & Configuration

In [ ]:
# Core imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.covariance import EllipticEnvelope
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("All imports successful!")

In [ ]:
# ============================================
# CONFIGURATION - MODIFY THESE VALUES
# ============================================

# Path to your full unified features CSV file
INPUT_FILE = "/home/smotaali/First_Full_Paper/dataset/phase1_raw_features/unified_full_rrc05_AS12880_2hop_2025-11-01_2025-11-30_5min.csv"

# Output directory (leave None to save in same directory as input)
OUTPUT_DIR = '/home/smotaali/First_Full_Paper/dataset/label_results_HDBSCAN'

# Estimated anomaly rate (0.1 = 10%)
# If you have no idea, start with 0.1
CONTAMINATION_ESTIMATE = 0.10  # Used only when ANOMALY_RATE_MODE == 'fixed'

# Label column name (if you have existing labels)
EXISTING_LABEL_COL = None  # Full datasets in /dataset do not have labels yet

# Time column name (for temporal analysis)
TIME_COL = 'window_start'  # Set to None if no time column

# Random seed for reproducibility
RANDOM_STATE = 42

# ============================================
# PLOT SAVING CONFIGURATION
# ============================================
SAVE_PLOTS = True  # Set to True to save all plots
PLOT_FORMAT = 'png'  # 'png', 'pdf', 'svg'
PLOT_DPI = 150  # Resolution (150 for screen, 300 for publication)

# Create output directory for plots
from pathlib import Path
input_path = Path(INPUT_FILE)
output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else input_path.parent
output_dir.mkdir(parents=True, exist_ok=True)
PLOT_DIR = output_dir / f"{input_path.stem}_plots"

if SAVE_PLOTS:
    PLOT_DIR.mkdir(exist_ok=True)
    print(f"Plots will be saved to: {PLOT_DIR}")

def save_plot(fig, name):
    """Helper function to save plots"""
    if SAVE_PLOTS:
        filepath = PLOT_DIR / f"{name}.{PLOT_FORMAT}"
        fig.savefig(filepath, dpi=PLOT_DPI, bbox_inches='tight')
        print(f"  Saved: {filepath}")

print(f"Configuration set:")
print(f"  Input file: {INPUT_FILE}")
print(f"  Fixed contamination reference: {CONTAMINATION_ESTIMATE*100:.0f}%")
print("  Anomaly rate mode: configured later in the hyperparameter cell")
print(f"  Save plots: {SAVE_PLOTS}")

## 2. Load and Explore Data

In [ ]:
# Load the data
df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df)} samples")
print(f"Columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")

In [ ]:
# Basic statistics
print("Data Overview:")
print("="*60)
df.info()
print("\n")
df.describe()

In [ ]:
# Check existing labels if present
if EXISTING_LABEL_COL and EXISTING_LABEL_COL in df.columns:
    print(f"Existing label distribution ({EXISTING_LABEL_COL}):")
    print("="*40)
    label_counts = df[EXISTING_LABEL_COL].value_counts()
    for label, count in label_counts.items():
        pct = count / len(df) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")
    
    # Visualize
    plt.figure(figsize=(10, 5))
    label_counts.plot(kind='bar', color=sns.color_palette("husl", len(label_counts)))
    plt.title('Existing Label Distribution')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No existing labels found - this is fine!")
    print("We will discover labels from the data.")

## 3. Prepare Features

In [ ]:
# Identify feature columns (exclude metadata only; keep the full feature set)
META_COLS = {
    'incident', 'window_start', 'window_end', 'window_id', 'timestamp', 'time',
    'label', 'label_rule', 'label_refined', 'label_discovered', 'discovered_label',
    'cluster', 'hdbscan_cluster', 'kmeans_cluster',
    'anomaly_score', 'anomaly_votes', 'consensus_score',
    'iso_forest_score', 'lof_score', 'statistical_score', 'elliptic_score', 'hdbscan_outlier_score',
    'source', 'collector', 'segment', 'asn'
}

# Get numeric columns that are not metadata
meta_cols_lower = {m.lower() for m in META_COLS}
candidate_cols = [c for c in df.columns if c.lower() not in meta_cols_lower]
feature_df = df[candidate_cols].select_dtypes(include=[np.number]).copy()
feature_cols = feature_df.columns.tolist()
constant_feature_cols = [c for c in feature_cols if feature_df[c].nunique(dropna=False) <= 1]

print(f"Identified {len(feature_cols)} feature columns:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1}. {col}")

if constant_feature_cols:
    print(f"\nConstant numeric columns detected (kept in this full-dataset run, but worth reviewing later): {constant_feature_cols}")

In [ ]:
# Prepare feature matrix
X = feature_df.values

# Handle missing and infinite values
valid_mask = np.isfinite(X).all(axis=1)
X_valid = X[valid_mask]

print(f"Valid samples: {len(X_valid)} / {len(X)}")
print(f"Excluded due to NaN/inf: {(~valid_mask).sum()}")

# Scale features (RobustScaler is robust to outliers)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_valid)

print(f"\nFeature matrix shape: {X_scaled.shape}")

## 4. Hyperparameter Tuning & Explanation

Before running anomaly detection, let's understand and tune the key parameters.

### Parameter Reference Table

| Parameter | Default | Range | How to Choose |
|-----------|---------|-------|---------------|
| `n_estimators` (IF) | 200 | 100-500 | More = stable but slower. 200 is usually enough |
| `n_neighbors` (LOF) | 20 | 5-50 | √n_samples or use tuning below |
| `IQR multiplier` | 1.5 | 1.5-3.0 | 1.5=standard, 3.0=very conservative |
| `PCA components` | 10 | 5-20 | Preserve ~95% variance |
| `DBSCAN eps` | auto | varies | Use k-distance graph |
| `min_samples` (DBSCAN) | 5 | 3-10 | Higher = fewer clusters |

In [ ]:
# ============================================
# HYPERPARAMETER CONFIGURATION
# ============================================
# These are the key parameters you can tune

# 0. Label-rate strategy
ANOMALY_RATE_MODE = 'natural'  # 'natural' lets methods choose their own rate when possible; 'fixed' uses CONTAMINATION_ESTIMATE
FIXED_CONTAMINATION = CONTAMINATION_ESTIMATE

# 1. Isolation Forest
N_ESTIMATORS = 200  # Number of trees (100-500, more = stable but slower)

# 2. Local Outlier Factor
# Rule of thumb: sqrt(n_samples) or between 10-50
N_NEIGHBORS_LOF = max(10, min(50, int(np.sqrt(len(X_valid)))))
LOF_NEIGHBOR_CANDIDATES = [5, 10, 15, 20, 30, 40, 50]

# 3. Statistical Outliers
IQR_MULTIPLIER = 1.5  # Standard=1.5, Conservative=2.0, Very Conservative=3.0
Z_SCORE_THRESHOLD = 3  # Standard=3, Conservative=2.5
STAT_MAD_MULTIPLIER = 3.0  # Used only when ANOMALY_RATE_MODE == 'natural'

# 4. PCA Components (for HDBSCAN / Elliptic Envelope)
# Will be set based on variance explained

# 5. HDBSCAN
MIN_SAMPLES_HDBSCAN = 5  # Minimum points to define local density
MIN_CLUSTER_SIZE_HDBSCAN = max(20, int(0.005 * len(X_valid)))
HDBSCAN_KNEE_SEARCH_FRAC = 0.30  # Search the upper score tail for the largest gap
HDBSCAN_TOP_GAPS_TO_REPORT = 2
HDBSCAN_OUTLIER_THRESHOLD_FALLBACK = 0.90  # Used only if the natural breakpoint cannot be found
HDBSCAN_UNSCORED_AS_SUSPICIOUS = True

# 6. Elliptic Envelope
# This estimator needs a contamination value even when we later threshold naturally via decision_function < 0.
ELLIPTIC_FIT_CONTAMINATION = min(FIXED_CONTAMINATION, 0.10)
ELLIPTIC_MIN_SAMPLE_FEATURE_RATIO = 5

# 7. Consensus / subtype characterization
USE_WEIGHTED_CONSENSUS = True
KMEANS_SMALL_CLUSTER_FRAC = 0.05
KMEANS_UPGRADE_UNCERTAIN = True

# 8. Stability analysis
BOOTSTRAP_STABILITY_ITERATIONS = 20
BOOTSTRAP_SAMPLE_FRAC = 0.80
BOOTSTRAP_RANDOM_STATE = RANDOM_STATE

# 9. Fourier template validation
FOURIER_BASELINE_SOURCE = 'robust_all'  # 'likely_normal' or 'robust_all'

print("Hyperparameters configured:")
print(f"  Anomaly rate mode: {ANOMALY_RATE_MODE}")
print(f"  Fixed contamination reference: {FIXED_CONTAMINATION*100:.1f}%")
print(f"  Isolation Forest n_estimators: {N_ESTIMATORS}")
print(f"  LOF initial n_neighbors: {N_NEIGHBORS_LOF} (based on sqrt({len(X_valid)}))")
print(f"  IQR multiplier: {IQR_MULTIPLIER}")
print(f"  Z-score threshold: {Z_SCORE_THRESHOLD}")
print(f"  Statistical MAD multiplier: {STAT_MAD_MULTIPLIER}")
print(f"  HDBSCAN min_cluster_size: {MIN_CLUSTER_SIZE_HDBSCAN}")
print(f"  HDBSCAN min_samples: {MIN_SAMPLES_HDBSCAN}")
print(f"  HDBSCAN knee search fraction: {HDBSCAN_KNEE_SEARCH_FRAC:.2f}")
print(f"  HDBSCAN top gaps reported: {HDBSCAN_TOP_GAPS_TO_REPORT}")
print(f"  HDBSCAN fallback threshold: {HDBSCAN_OUTLIER_THRESHOLD_FALLBACK}")
print(f"  HDBSCAN unscored as suspicious: {HDBSCAN_UNSCORED_AS_SUSPICIOUS}")
print(f"  Elliptic fit contamination: {ELLIPTIC_FIT_CONTAMINATION}")
print(f"  Weighted consensus: {USE_WEIGHTED_CONSENSUS}")
print(f"  K-Means uncertain-label upgrade: {KMEANS_UPGRADE_UNCERTAIN}")
print(f"  Bootstrap stability runs: {BOOTSTRAP_STABILITY_ITERATIONS} at {BOOTSTRAP_SAMPLE_FRAC*100:.0f}% sample fraction")
print(f"  Fourier baseline source: {FOURIER_BASELINE_SOURCE}")


In [ ]:
# Tune LOF n_neighbors using local stability rather than a target anomaly rate
print("Tuning LOF n_neighbors...")

neighbor_range = [n for n in LOF_NEIGHBOR_CANDIDATES if n < len(X_valid) // 2]
lof_fit_contamination = 'auto' if ANOMALY_RATE_MODE == 'natural' else FIXED_CONTAMINATION

lof_stability = []
for n in neighbor_range:
    lof_temp = LocalOutlierFactor(
        n_neighbors=n,
        contamination=lof_fit_contamination,
        novelty=False
    )
    temp_predictions = lof_temp.fit_predict(X_scaled)
    temp_scores = lof_temp.negative_outlier_factor_

    if ANOMALY_RATE_MODE == 'natural':
        temp_anomalies = temp_scores < lof_temp.offset_
    else:
        temp_anomalies = temp_predictions == -1

    anomaly_rate = temp_anomalies.mean()
    lof_stability.append({'n': n, 'anomaly_rate': anomaly_rate})
    print(f"  n_neighbors={n}: {anomaly_rate*100:.2f}% anomalies")

if len(lof_stability) >= 2:
    stability_scores = []
    for idx, item in enumerate(lof_stability):
        diffs = []
        if idx > 0:
            diffs.append(abs(item['anomaly_rate'] - lof_stability[idx - 1]['anomaly_rate']))
        if idx < len(lof_stability) - 1:
            diffs.append(abs(item['anomaly_rate'] - lof_stability[idx + 1]['anomaly_rate']))
        stability_score = float(np.mean(diffs)) if diffs else 0.0
        stability_scores.append((item['n'], stability_score))
else:
    stability_scores = [(lof_stability[0]['n'], 0.0)]

plt.figure(figsize=(10, 4))
ns = [item['n'] for item in lof_stability]
rates = [item['anomaly_rate'] * 100 for item in lof_stability]
plt.plot(ns, rates, 'bo-', markersize=8)
if ANOMALY_RATE_MODE == 'fixed':
    plt.axhline(
        y=FIXED_CONTAMINATION * 100,
        color='red',
        linestyle='--',
        label=f'Fixed target: {FIXED_CONTAMINATION*100:.1f}%'
    )
plt.xlabel('n_neighbors')
plt.ylabel('Anomaly Rate (%)')
plt.title('LOF Sensitivity to n_neighbors\n(choose the most stable local region)')
if ANOMALY_RATE_MODE == 'fixed':
    plt.legend()
plt.grid(True)
plt.show()

best_n = min(stability_scores, key=lambda x: (x[1], abs(x[0] - N_NEIGHBORS_LOF)))[0]
N_NEIGHBORS_LOF = best_n

print("\nLocal stability scores (smaller is better):")
for n, score in stability_scores:
    print(f"  n_neighbors={n}: local change score = {score:.6f}")

print(f"\nRecommended n_neighbors: {best_n}")
print(f"Using LOF n_neighbors={N_NEIGHBORS_LOF} for final model")


In [ ]:
# Determine optimal PCA components (preserve 95% variance)
print("Analyzing PCA variance...")

pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1

print(f"  Components for 90% variance: {n_components_90}")
print(f"  Components for 95% variance: {n_components_95}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1), pca_full.explained_variance_ratio_ * 100)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Variance per Component')

axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance * 100, 'b-', linewidth=2)
axes[1].axhline(y=95, color='red', linestyle='--', label='95% threshold')
axes[1].axhline(y=90, color='orange', linestyle='--', label='90% threshold')
axes[1].axvline(x=n_components_95, color='red', linestyle=':', alpha=0.5)
axes[1].axvline(x=n_components_90, color='orange', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

N_PCA_COMPONENTS = n_components_95
print(f"\nUsing {N_PCA_COMPONENTS} PCA components (95% variance)")

pca = PCA(n_components=N_PCA_COMPONENTS)
X_pca = pca.fit_transform(X_scaled)
PCA_VARIANCE_EXPLAINED = float(np.sum(pca.explained_variance_ratio_))
print(f"Prepared PCA feature matrix: {X_pca.shape} ({PCA_VARIANCE_EXPLAINED*100:.1f}% variance)")


## 5. Anomaly Detection Methods (with tuned parameters)


In [ ]:
# Method 1: Isolation Forest (using tuned parameters)
print("[1/5] Running Isolation Forest...")

iso_contamination = 'auto' if ANOMALY_RATE_MODE == 'natural' else FIXED_CONTAMINATION
iso_forest = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=iso_contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

iso_forest.fit(X_scaled)
iso_scores = iso_forest.decision_function(X_scaled)

if ANOMALY_RATE_MODE == 'natural':
    iso_anomalies = iso_scores < 0
    iso_threshold_desc = 'natural (decision_function < 0)'
else:
    iso_predictions = iso_forest.predict(X_scaled)
    iso_anomalies = iso_predictions == -1
    iso_threshold_desc = f'fixed contamination={FIXED_CONTAMINATION:.3f}'

print(f"  Parameters: n_estimators={N_ESTIMATORS}, contamination={iso_contamination}")
print(f"  Threshold mode: {iso_threshold_desc}")
print(f"  Found {iso_anomalies.sum()} anomalies ({iso_anomalies.mean()*100:.2f}%)")


In [ ]:
# Method 2: Local Outlier Factor (using tuned parameters)
print("[2/5] Running Local Outlier Factor...")

lof_contamination = 'auto' if ANOMALY_RATE_MODE == 'natural' else FIXED_CONTAMINATION
lof = LocalOutlierFactor(
    n_neighbors=N_NEIGHBORS_LOF,
    contamination=lof_contamination,
    novelty=False
)

lof_predictions = lof.fit_predict(X_scaled)
lof_scores = lof.negative_outlier_factor_

if ANOMALY_RATE_MODE == 'natural':
    lof_anomalies = lof_scores < lof.offset_
    lof_threshold_desc = f'natural (negative_outlier_factor_ < offset={lof.offset_:.4f})'
else:
    lof_anomalies = lof_predictions == -1
    lof_threshold_desc = f'fixed contamination={FIXED_CONTAMINATION:.3f}'

print(f"  Parameters: n_neighbors={N_NEIGHBORS_LOF}, contamination={lof_contamination}")
print(f"  Threshold mode: {lof_threshold_desc}")
print(f"  Found {lof_anomalies.sum()} anomalies ({lof_anomalies.mean()*100:.2f}%)")


In [ ]:
# Method 3: Statistical Analysis (using tuned parameters)
print("[3/5] Running Statistical Analysis...")

def statistical_outlier_scores(X, z_threshold=3, iqr_mult=1.5):
    # Calculate outlier scores using Z-score and IQR on the raw feature values.
    n_samples, n_features = X.shape
    outlier_scores = np.zeros(n_samples)

    for i in range(n_features):
        col = X[:, i]
        z_scores = np.abs(stats.zscore(col, nan_policy='omit'))
        z_outliers = z_scores > z_threshold

        Q1, Q3 = np.percentile(col, [25, 75])
        IQR = Q3 - Q1
        iqr_outliers = (col < Q1 - iqr_mult * IQR) | (col > Q3 + iqr_mult * IQR)

        outlier_scores += z_outliers.astype(float) + iqr_outliers.astype(float)

    return outlier_scores / (2 * n_features)

# Intentionally use raw feature values here: z-score and IQR are distribution-aware rules,
# and applying them after RobustScaler would make the IQR rule largely redundant.
stat_scores = statistical_outlier_scores(
    X_valid,
    z_threshold=Z_SCORE_THRESHOLD,
    iqr_mult=IQR_MULTIPLIER
)

if ANOMALY_RATE_MODE == 'natural':
    stat_median = float(np.median(stat_scores))
    stat_mad = float(np.median(np.abs(stat_scores - stat_median)))
    if stat_mad > 0:
        stat_threshold = stat_median + STAT_MAD_MULTIPLIER * 1.4826 * stat_mad
    else:
        stat_threshold = stat_median + STAT_MAD_MULTIPLIER * float(np.std(stat_scores))

    if (not np.isfinite(stat_threshold)) or (stat_threshold <= stat_median):
        stat_threshold = float(np.quantile(stat_scores, 0.95))

    stat_threshold_desc = f'natural (median + {STAT_MAD_MULTIPLIER} * MAD)'
else:
    stat_threshold = float(np.percentile(stat_scores, 100 * (1 - FIXED_CONTAMINATION)))
    stat_threshold_desc = f'fixed percentile at top {FIXED_CONTAMINATION*100:.1f}%'

stat_anomalies = stat_scores > stat_threshold

print(f"  Parameters: Z-score threshold={Z_SCORE_THRESHOLD}, IQR multiplier={IQR_MULTIPLIER}")
print(f"  Threshold mode: {stat_threshold_desc}")
print(f"  Statistical score threshold: {stat_threshold:.4f}")
print(f"  Found {stat_anomalies.sum()} anomalies ({stat_anomalies.mean()*100:.2f}%)")


In [ ]:
# Method 4: Elliptic Envelope (Robust Covariance)
print("[4/5] Running Elliptic Envelope...")

elliptic_input = X_pca
elliptic_ratio = len(X_valid) / max(1, elliptic_input.shape[1])
print(f"  Sample/feature ratio on PCA space: {elliptic_ratio:.1f}")

try:
    if elliptic_ratio <= ELLIPTIC_MIN_SAMPLE_FEATURE_RATIO:
        raise ValueError(
            f"sample/feature ratio {elliptic_ratio:.1f} <= {ELLIPTIC_MIN_SAMPLE_FEATURE_RATIO}; covariance would be unstable"
        )

    elliptic = EllipticEnvelope(
        contamination=ELLIPTIC_FIT_CONTAMINATION,
        random_state=RANDOM_STATE
    )
    elliptic.fit(elliptic_input)
    elliptic_predictions = elliptic.predict(elliptic_input)
    elliptic_scores = elliptic.decision_function(elliptic_input)

    if ANOMALY_RATE_MODE == 'natural':
        elliptic_anomalies = elliptic_scores < 0
        elliptic_threshold_desc = 'natural (decision_function < 0)'
    else:
        elliptic_anomalies = elliptic_predictions == -1
        elliptic_threshold_desc = f'fixed fit contamination={ELLIPTIC_FIT_CONTAMINATION:.3f}'

    print(f"  Threshold mode: {elliptic_threshold_desc}")
    print(f"  Found {elliptic_anomalies.sum()} anomalies ({elliptic_anomalies.mean()*100:.2f}%)")
    elliptic_success = True
except Exception as e:
    print(f"  Skipped with warning: {e}")
    elliptic_anomalies = np.zeros(len(X_valid), dtype=bool)
    elliptic_scores = np.zeros(len(X_valid))
    elliptic_success = False


In [ ]:
# Method 5: HDBSCAN (using outlier_scores, not only cluster noise)
print("[5/5] Running HDBSCAN Clustering...")

try:
    import hdbscan
except ImportError:
    print("Installing hdbscan...")
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'hdbscan', '-q'])
    import hdbscan

import gc

print(f"  Reusing PCA feature matrix: {X_pca.shape} ({PCA_VARIANCE_EXPLAINED*100:.1f}% variance)")

MIN_CLUSTER_SIZE = MIN_CLUSTER_SIZE_HDBSCAN
MIN_SAMPLES = MIN_SAMPLES_HDBSCAN

print(f"  Running HDBSCAN on {len(X_pca)} samples...")
print(f"  Parameters: min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_method='leaf',
    prediction_data=False,
    core_dist_n_jobs=-1
)

hdbscan_labels = clusterer.fit_predict(X_pca)
raw_outlier_scores = np.asarray(clusterer.outlier_scores_, dtype=float)
hdbscan_unscored_mask = ~np.isfinite(raw_outlier_scores)
finite_outlier_scores = raw_outlier_scores[np.isfinite(raw_outlier_scores)]
outlier_scores = raw_outlier_scores.copy()

gc.collect()

n_clusters = len(set(hdbscan_labels)) - (1 if -1 in hdbscan_labels else 0)
noise_count = int((hdbscan_labels == -1).sum())
unscored_count = int(hdbscan_unscored_mask.sum())
print(f"  Found {n_clusters} clusters, {noise_count} noise points")
print(f"  Unscored HDBSCAN points: {unscored_count}")

def detect_hdbscan_score_breakpoint(scores, search_fraction=0.30, top_k=2):
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]
    if len(scores) < 10 or np.ptp(scores) <= 1e-12:
        return None

    sorted_desc = np.sort(scores)[::-1]
    search_len = max(5, int(np.ceil(len(sorted_desc) * search_fraction)))
    search_len = min(search_len, len(sorted_desc) - 1)
    upper_scores = sorted_desc[:search_len + 1]
    diffs = upper_scores[:-1] - upper_scores[1:]

    gap_order = np.argsort(diffs)[::-1][:max(1, min(top_k, len(diffs)))]
    top_gaps = []
    for gap_idx in gap_order:
        gap_size = float(diffs[gap_idx])
        top_gaps.append({
            'gap_idx': int(gap_idx),
            'gap_size': gap_size,
            'upper_score': float(upper_scores[gap_idx]),
            'lower_score': float(upper_scores[gap_idx + 1]),
            'threshold': float((upper_scores[gap_idx] + upper_scores[gap_idx + 1]) / 2.0),
        })

    if not top_gaps or top_gaps[0]['gap_size'] <= 0:
        return None

    best_gap = top_gaps[0]
    return {
        'threshold': best_gap['threshold'],
        'gap_idx': best_gap['gap_idx'],
        'gap_size': best_gap['gap_size'],
        'sorted_scores': sorted_desc,
        'search_len': search_len,
        'top_gaps': top_gaps,
    }

knee_result = None
if ANOMALY_RATE_MODE == 'natural':
    knee_result = detect_hdbscan_score_breakpoint(
        finite_outlier_scores,
        search_fraction=HDBSCAN_KNEE_SEARCH_FRAC,
        top_k=HDBSCAN_TOP_GAPS_TO_REPORT
    )
    if knee_result is not None:
        outlier_threshold = knee_result['threshold']
        threshold_desc = (
            f"natural largest-gap breakpoint in the top {HDBSCAN_KNEE_SEARCH_FRAC*100:.0f}% "
            f"of HDBSCAN scores (gap={knee_result['gap_size']:.4f})"
        )
    else:
        outlier_threshold = HDBSCAN_OUTLIER_THRESHOLD_FALLBACK
        threshold_desc = f"fallback natural threshold > {HDBSCAN_OUTLIER_THRESHOLD_FALLBACK:.2f}"
else:
    outlier_threshold = float(np.percentile(finite_outlier_scores, 100 * (1 - FIXED_CONTAMINATION)))
    threshold_desc = f'fixed percentile at top {FIXED_CONTAMINATION*100:.1f}%'

hdbscan_anomalies = np.zeros(len(outlier_scores), dtype=bool)
finite_mask = np.isfinite(outlier_scores)
hdbscan_anomalies[finite_mask] = outlier_scores[finite_mask] > outlier_threshold
if HDBSCAN_UNSCORED_AS_SUSPICIOUS and hdbscan_unscored_mask.any():
    hdbscan_anomalies[hdbscan_unscored_mask] = True

anomaly_rate = hdbscan_anomalies.mean() * 100

print("\n  Using outlier_scores for anomaly detection:")
print(f"  Threshold mode: {threshold_desc}")
print(f"  Outlier score threshold: {outlier_threshold:.3f}")
if knee_result is not None:
    print("  Top HDBSCAN score gaps considered:")
    for gap in knee_result['top_gaps']:
        print(
            f"    rank {gap['gap_idx'] + 1}: gap={gap['gap_size']:.4f}, "
            f"candidate threshold={gap['threshold']:.3f}"
        )
if HDBSCAN_UNSCORED_AS_SUSPICIOUS and hdbscan_unscored_mask.any():
    print(f"  Added {unscored_count} unscored points as suspicious HDBSCAN anomalies")
print(f"  Anomalies detected: {hdbscan_anomalies.sum()} ({anomaly_rate:.2f}%)")

dbscan_anomalies = hdbscan_anomalies
dbscan_labels = hdbscan_labels
hdbscan_outlier_scores = outlier_scores

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].hist(finite_outlier_scores, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=outlier_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Threshold: {outlier_threshold:.3f}')
axes[0].set_xlabel('HDBSCAN Outlier Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Outlier Score Distribution\n({threshold_desc})')
axes[0].legend()

sorted_scores = np.sort(finite_outlier_scores)[::-1]
axes[1].plot(range(1, len(sorted_scores) + 1), sorted_scores, color='navy', linewidth=2)
axes[1].axhline(y=outlier_threshold, color='red', linestyle='--', linewidth=1.5)
if knee_result is not None:
    gap_colors = ['orange', 'gold']
    for gap_idx, gap in enumerate(knee_result['top_gaps']):
        color = gap_colors[min(gap_idx, len(gap_colors) - 1)]
        axes[1].axvline(x=gap['gap_idx'] + 1, color=color, linestyle=':', linewidth=1.3)
        axes[1].scatter(
            [gap['gap_idx'] + 1],
            [sorted_scores[gap['gap_idx']]],
            color=color,
            s=40,
            label='largest gap' if gap_idx == 0 else f'gap #{gap_idx + 1}'
        )
axes[1].set_xlabel('Sorted sample rank (descending score)')
axes[1].set_ylabel('Outlier score')
axes[1].set_title('Natural Breakpoint in HDBSCAN Scores')
axes[1].grid(True, alpha=0.3)
if knee_result is not None:
    axes[1].legend()

cluster_counts = pd.Series(hdbscan_labels).value_counts().sort_index()
colors_bar = ['red' if idx == -1 else 'steelblue' for idx in cluster_counts.index]
axes[2].bar(range(len(cluster_counts)), cluster_counts.values, color=colors_bar)
axes[2].set_xlabel('Cluster ID (-1 = noise)')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Cluster Sizes ({n_clusters} clusters + noise)')

plt.tight_layout()
save_plot(fig, '06_hdbscan_results')
plt.show()


## 6. Compute Consensus

In [ ]:
# Collect method decisions and compute consensus
methods_dict = {
    'Isolation Forest': np.asarray(iso_anomalies, dtype=bool),
    'Local Outlier Factor': np.asarray(lof_anomalies, dtype=bool),
    'Statistical': np.asarray(stat_anomalies, dtype=bool),
    'HDBSCAN': np.asarray(dbscan_anomalies, dtype=bool),
}
if elliptic_success:
    methods_dict['Elliptic Envelope'] = np.asarray(elliptic_anomalies, dtype=bool)

methods_df = pd.DataFrame(methods_dict)

def derive_consensus_thresholds(weight_series):
    weights = np.sort(np.asarray(weight_series, dtype=float))
    n = len(weights)
    if n == 0:
        return {
            'uncertain': 1.0,
            'likely_anomaly': 1.0,
            'high_confidence_anomaly': 1.0,
            'min_votes_for_likely': 0,
            'effective_methods': 0,
        }

    uncertain_threshold = float(weights[0])
    min_votes_for_likely = min(n, max(2, int(np.ceil((n + 1) / 2))))
    likely_threshold = float(weights[:min_votes_for_likely].sum())

    if n <= 2:
        high_conf_threshold = 1.0
    else:
        high_conf_threshold = float(1.0 - weights[-1])

    high_conf_threshold = float(max(high_conf_threshold, likely_threshold))
    high_conf_threshold = float(min(1.0, high_conf_threshold))

    return {
        'uncertain': uncertain_threshold,
        'likely_anomaly': likely_threshold,
        'high_confidence_anomaly': high_conf_threshold,
        'min_votes_for_likely': min_votes_for_likely,
        'effective_methods': n,
    }

def labels_from_scores(scores, thresholds):
    return np.where(
        scores >= thresholds['high_confidence_anomaly'], 'high_confidence_anomaly',
        np.where(
            scores >= thresholds['likely_anomaly'], 'likely_anomaly',
            np.where(
                scores >= thresholds['uncertain'], 'uncertain',
                'likely_normal'
            )
        )
    )

def apply_consensus_state(method_frame, method_weights_full, thresholds):
    weight_series = method_weights_full.reindex(method_frame.columns).fillna(0.0).astype(float)
    if (not np.isfinite(weight_series).all()) or weight_series.sum() <= 0:
        weight_series = pd.Series(1.0 / len(method_frame.columns), index=method_frame.columns)
    else:
        weight_series = weight_series / weight_series.sum()

    score = (method_frame.astype(float) * weight_series.values).sum(axis=1).values
    labels = labels_from_scores(score, thresholds)
    return score, labels

def compute_consensus_state(method_frame, use_weighted=True):
    method_frame = method_frame.astype(bool).copy()
    raw_anomaly_votes = method_frame.sum(axis=1).astype(int).values
    equal_consensus = method_frame.mean(axis=1).values

    active_methods = [
        col for col in method_frame.columns
        if method_frame[col].nunique(dropna=False) > 1
    ]
    inactive_methods = [col for col in method_frame.columns if col not in active_methods]

    if not active_methods:
        active_methods = list(method_frame.columns)
        inactive_methods = []

    effective_df = method_frame[active_methods]
    effective_method_count = len(effective_df.columns)
    correlation = effective_df.astype(float).corr().replace([np.inf, -np.inf], np.nan)

    if use_weighted and effective_method_count > 1:
        mean_abs_corr = (correlation.abs().sum(axis=1) - 1.0) / (effective_method_count - 1)
        mean_abs_corr = mean_abs_corr.replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(lower=0.0)
        weights_active = 1.0 / (1.0 + mean_abs_corr)
        if (not np.isfinite(weights_active).all()) or weights_active.sum() <= 0:
            weights_active = pd.Series(1.0, index=effective_df.columns)
        weights_active = weights_active / weights_active.sum()
        consensus_mode_local = 'weighted'
    else:
        weights_active = pd.Series(1.0 / effective_method_count, index=effective_df.columns)
        consensus_mode_local = 'equal'

    weights_full = pd.Series(0.0, index=method_frame.columns)
    weights_full.loc[weights_active.index] = weights_active
    thresholds = derive_consensus_thresholds(weights_active)
    consensus_local, labels_local = apply_consensus_state(method_frame, weights_full, thresholds)

    return {
        'raw_anomaly_votes': raw_anomaly_votes,
        'equal_consensus_score': equal_consensus,
        'consensus_score': consensus_local,
        'labels': labels_local,
        'method_weights': weights_full,
        'active_methods': active_methods,
        'inactive_methods': inactive_methods,
        'effective_method_count': effective_method_count,
        'correlation': correlation.fillna(0.0),
        'thresholds': thresholds,
        'mode': consensus_mode_local,
    }

consensus_state = compute_consensus_state(methods_df, use_weighted=USE_WEIGHTED_CONSENSUS)
anomaly_votes = consensus_state['raw_anomaly_votes']
equal_consensus_score = consensus_state['equal_consensus_score']
consensus_score = consensus_state['consensus_score']
discovered_labels = consensus_state['labels']
method_weights = consensus_state['method_weights']
active_methods = consensus_state['active_methods']
inactive_methods = consensus_state['inactive_methods']
effective_method_count = consensus_state['effective_method_count']
correlation = consensus_state['correlation']
consensus_thresholds = consensus_state['thresholds']
consensus_mode = consensus_state['mode']
discovered_labels_pre_kmeans = discovered_labels.copy()
kmeans_label_boosted = np.zeros(len(discovered_labels), dtype=bool)

print("Consensus Results:")
print("=" * 60)
print(f"  Consensus mode: {consensus_mode}")
print(f"  Effective methods used for weighting: {effective_method_count}/{len(methods_df.columns)}")
print("  Method weights:")
for method_name, weight in method_weights.sort_values(ascending=False).items():
    print(f"    {method_name:<22} {weight:.3f}")

if inactive_methods:
    print(f"  Zero-variance methods excluded from weighting: {inactive_methods}")

print("  Dynamic thresholds:")
print(f"    uncertain: >= {consensus_thresholds['uncertain']:.3f}")
print(
    f"    likely_anomaly: >= {consensus_thresholds['likely_anomaly']:.3f} "
    f"(strict majority of {consensus_thresholds['min_votes_for_likely']} effective methods)"
)
print(
    f"    high_confidence_anomaly: >= {consensus_thresholds['high_confidence_anomaly']:.3f} "
    f"(all but maybe one effective method)"
)

for label in ['likely_normal', 'uncertain', 'likely_anomaly', 'high_confidence_anomaly']:
    count = int((discovered_labels == label).sum())
    pct = count / len(discovered_labels) * 100
    marker = '[normal]' if 'normal' in label else '[anomaly]' if 'anomaly' in label else '[uncertain]'
    print(f"  {marker:<11} {label}: {count} ({pct:.1f}%)")


In [ ]:
# Visualize consensus distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of consensus scores
axes[0].hist(consensus_score, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel(f'Consensus Score ({consensus_mode})')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Consensus Scores')
axes[0].axvline(x=consensus_thresholds['uncertain'], color='#f1c40f', linestyle='--', label='uncertain threshold')
axes[0].axvline(x=consensus_thresholds['likely_anomaly'], color='#e67e22', linestyle='--', label='likely anomaly threshold')
axes[0].axvline(x=consensus_thresholds['high_confidence_anomaly'], color='#c0392b', linestyle='--', label='high-confidence threshold')
axes[0].legend()

# Pie chart of discovered labels
label_counts = pd.Series(discovered_labels).value_counts()
colors = {'likely_normal': '#2ecc71', 'uncertain': '#f1c40f',
          'likely_anomaly': '#e74c3c', 'high_confidence_anomaly': '#c0392b'}
pie_colors = [colors[l] for l in label_counts.index]
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=pie_colors, startangle=90)
axes[1].set_title('Consensus Label Distribution')

plt.tight_layout()
save_plot(fig, '01_consensus_distribution')
plt.show()


## 7. Method Agreement Analysis

In [ ]:
# Visualize method agreement and the weights used in the consensus
fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1.8, 1.0]})

sns.heatmap(correlation, annot=True, cmap='RdYlGn_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Agreement Between Active Anomaly Detection Methods')

method_weights.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlim(0, max(0.4, method_weights.max() * 1.15))
axes[1].set_xlabel('Weight')
axes[1].set_title(f'Consensus Weights ({consensus_mode})')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
save_plot(fig, '02_method_agreement_heatmap')
plt.show()

print("\nPer-method anomaly rates:")
for col in methods_df.columns:
    rate = methods_df[col].mean() * 100
    active_tag = 'active' if col in active_methods else 'dropped'
    print(f"  {col}: {rate:.2f}% (weight={method_weights[col]:.3f}, {active_tag})")


## 8. K-Means Subtype Characterization & Label Refinement

K-Means is used here as supplementary evidence. Small subtype clusters can upgrade borderline `uncertain` windows into `likely_anomaly`, while still preserving the original consensus-only label for auditability.


In [ ]:
# Find optimal number of clusters using silhouette score
print("Finding optimal number of clusters...")

max_clusters = min(10, len(X_scaled) - 1)
silhouette_scores = []

for k in range(2, max_clusters + 1):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append((k, score))
    print(f"  k={k}: silhouette score = {score:.4f}")

optimal_k = max(silhouette_scores, key=lambda x: x[1])[0]
print(f"\nOptimal number of clusters: {optimal_k}")

# Plot silhouette scores
fig, ax = plt.subplots(figsize=(10, 5))
ks, scores = zip(*silhouette_scores)
ax.plot(ks, scores, 'bo-', markersize=10)
ax.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs Number of Clusters')
ax.legend()
ax.grid(True)
save_plot(fig, '03_silhouette_scores')
plt.show()

In [ ]:
# Perform final clustering with optimal k
kmeans_final = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
small_cluster_ids = cluster_counts[cluster_counts / len(cluster_labels) < KMEANS_SMALL_CLUSTER_FRAC].index.tolist()
kmeans_small_cluster_flag = np.isin(cluster_labels, small_cluster_ids)

print("Cluster distribution:")
for i in range(optimal_k):
    count = int(cluster_counts.get(i, 0))
    pct = count / len(cluster_labels) * 100
    print(f"  Cluster {i}: {count} samples ({pct:.1f}%)")

if small_cluster_ids:
    print(f"\nSmall K-Means clusters flagged for subtype review (< {KMEANS_SMALL_CLUSTER_FRAC*100:.1f}% of data): {small_cluster_ids}")
    print(f"  Windows in small clusters: {kmeans_small_cluster_flag.sum()} ({kmeans_small_cluster_flag.mean()*100:.2f}%)")
else:
    print(f"\nNo K-Means clusters were smaller than {KMEANS_SMALL_CLUSTER_FRAC*100:.1f}% of the dataset.")

discovered_labels_pre_kmeans = discovered_labels.copy()
if KMEANS_UPGRADE_UNCERTAIN:
    kmeans_label_boosted = (discovered_labels == 'uncertain') & kmeans_small_cluster_flag
    if kmeans_label_boosted.any():
        discovered_labels = discovered_labels.copy()
        discovered_labels[kmeans_label_boosted] = 'likely_anomaly'
        print(
            f"\nUpgraded {kmeans_label_boosted.sum()} uncertain windows to likely_anomaly "
            f"because they fall in small K-Means subtypes."
        )
    else:
        print("\nNo uncertain windows needed a K-Means subtype upgrade.")
else:
    kmeans_label_boosted = np.zeros(len(discovered_labels), dtype=bool)
    print("\nK-Means subtype upgrades are disabled.")

print("\nFinal label distribution after K-Means refinement:")
for label in ['likely_normal', 'uncertain', 'likely_anomaly', 'high_confidence_anomaly']:
    count = int((discovered_labels == label).sum())
    pct = count / len(discovered_labels) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")


In [ ]:
# Visualize clusters using PCA
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: K-Means subtype view
scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels,
                           cmap='tab10', alpha=0.6, s=30)
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('K-Means Subtypes (descriptive only)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: Discovered anomalies
colors = {'likely_normal': '#2ecc71', 'uncertain': '#f1c40f',
          'likely_anomaly': '#e74c3c', 'high_confidence_anomaly': '#c0392b'}
for label in ['likely_normal', 'uncertain', 'likely_anomaly', 'high_confidence_anomaly']:
    mask = discovered_labels == label
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=colors[label],
                    label=label, alpha=0.6, s=30)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Discovered Anomalies')
axes[1].legend()

plt.tight_layout()
save_plot(fig, '04_pca_clusters_anomalies')
plt.show()


## 9. Bootstrap Label Stability

This block is intentionally a subsampling stability test, not a classical bootstrap confidence interval: each run uses 80% of the rows without replacement to test how sensitive the consensus weighting and final labels are to sample composition.


In [ ]:
# Bootstrap label stability using the existing method outputs (no base-method refit)
print("Running bootstrap label stability check...")

rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
sample_size = max(2, int(np.ceil(BOOTSTRAP_SAMPLE_FRAC * len(methods_df))))
bootstrap_label_matrix = np.empty((len(methods_df), BOOTSTRAP_STABILITY_ITERATIONS), dtype=object)

for b in range(BOOTSTRAP_STABILITY_ITERATIONS):
    subset_idx = rng.choice(methods_df.index.to_numpy(), size=sample_size, replace=False)
    bootstrap_state = compute_consensus_state(
        methods_df.loc[subset_idx],
        use_weighted=USE_WEIGHTED_CONSENSUS
    )
    _, boot_labels = apply_consensus_state(
        methods_df,
        bootstrap_state['method_weights'],
        bootstrap_state['thresholds']
    )

    if KMEANS_UPGRADE_UNCERTAIN:
        boost_mask = (boot_labels == 'uncertain') & kmeans_small_cluster_flag
        boot_labels = boot_labels.copy()
        boot_labels[boost_mask] = 'likely_anomaly'

    bootstrap_label_matrix[:, b] = boot_labels

    if (b + 1) in {1, BOOTSTRAP_STABILITY_ITERATIONS} or (b + 1) % 5 == 0:
        print(f"  completed {b + 1}/{BOOTSTRAP_STABILITY_ITERATIONS} bootstrap runs")

label_stability_score = (bootstrap_label_matrix == discovered_labels[:, None]).mean(axis=1)
label_flip_rate = 1.0 - label_stability_score
bootstrap_modal_label = np.empty(len(methods_df), dtype=object)
bootstrap_modal_agreement = np.zeros(len(methods_df), dtype=float)

for i in range(len(methods_df)):
    counts = pd.Series(bootstrap_label_matrix[i, :]).value_counts()
    bootstrap_modal_label[i] = counts.index[0]
    bootstrap_modal_agreement[i] = counts.iloc[0] / BOOTSTRAP_STABILITY_ITERATIONS

print(f"\nAverage stability score: {label_stability_score.mean():.3f}")
print(f"Median stability score:  {np.median(label_stability_score):.3f}")
print(f"Low-stability windows (<0.60): {(label_stability_score < 0.60).sum()} / {len(label_stability_score)}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(label_stability_score, bins=np.linspace(0, 1, 21), edgecolor='black', color='slateblue', alpha=0.8)
axes[0].axvline(label_stability_score.mean(), color='crimson', linestyle='--', linewidth=2,
                label=f"mean = {label_stability_score.mean():.3f}")
axes[0].set_xlabel('Label Stability Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Bootstrap Stability of Final Labels')
axes[0].legend()

stability_summary = (
    pd.DataFrame({'label': discovered_labels, 'stability': label_stability_score})
    .groupby('label')['stability']
    .agg(['mean', 'median', 'count'])
    .reindex(['likely_normal', 'uncertain', 'likely_anomaly', 'high_confidence_anomaly'])
    .fillna(0)
)
axes[1].bar(stability_summary.index, stability_summary['mean'], color=['#2ecc71', '#f1c40f', '#e67e22', '#c0392b'])
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('Mean stability')
axes[1].set_title('Mean Stability by Final Label')
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_plot(fig, '05_label_stability')
plt.show()

leave_one_method_out_rows = []
for dropped_method in methods_df.columns:
    reduced_methods_df = methods_df.drop(columns=[dropped_method])
    loo_state = compute_consensus_state(
        reduced_methods_df,
        use_weighted=USE_WEIGHTED_CONSENSUS
    )
    _, loo_labels = apply_consensus_state(
        reduced_methods_df,
        loo_state['method_weights'],
        loo_state['thresholds']
    )

    if KMEANS_UPGRADE_UNCERTAIN:
        boost_mask = (loo_labels == 'uncertain') & kmeans_small_cluster_flag
        loo_labels = loo_labels.copy()
        loo_labels[boost_mask] = 'likely_anomaly'

    changed_mask = loo_labels != discovered_labels
    leave_one_method_out_rows.append({
        'dropped_method': dropped_method,
        'changed_windows': int(changed_mask.sum()),
        'changed_pct': float(changed_mask.mean() * 100),
        'agreement_pct': float((~changed_mask).mean() * 100),
    })

leave_one_method_out_df = pd.DataFrame(leave_one_method_out_rows).sort_values('changed_pct', ascending=False)
print("\nLeave-one-method-out sensitivity:")
print(
    leave_one_method_out_df.to_string(
        index=False,
        formatters={
            'changed_pct': lambda v: f"{v:.2f}",
            'agreement_pct': lambda v: f"{v:.2f}",
        }
    )
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(leave_one_method_out_df['dropped_method'], leave_one_method_out_df['changed_pct'], color='salmon')
ax.invert_yaxis()
ax.set_xlabel('Changed labels (%)')
ax.set_ylabel('Dropped method')
ax.set_title('Leave-One-Method-Out Label Sensitivity')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_plot(fig, '05b_leave_one_method_out')
plt.show()

stability_report_cols = []
if TIME_COL and TIME_COL in df.columns:
    stability_report_cols.append(TIME_COL)

stability_report = df.loc[valid_mask, stability_report_cols].copy()
stability_report['final_label'] = discovered_labels
stability_report['stability_score'] = label_stability_score
stability_report['flip_rate'] = label_flip_rate
stability_report['bootstrap_modal_label'] = bootstrap_modal_label

top_unstable = stability_report.sort_values(['stability_score', 'flip_rate']).head(10)
print("\nMost unstable windows:")
try:
    from IPython.display import display as ipy_display
    ipy_display(top_unstable)
except Exception:
    print(top_unstable.to_string(index=False))


## 10. Feature Analysis: What Makes Anomalies Different?

In [ ]:
# Compare features between normal and anomalous samples
normal_mask = discovered_labels == 'likely_normal'
anomaly_mask = np.isin(discovered_labels, ['high_confidence_anomaly', 'likely_anomaly'])

feature_analysis = []

for i, col in enumerate(feature_cols):
    normal_vals = X_valid[normal_mask, i]
    anomaly_vals = X_valid[anomaly_mask, i]

    if len(normal_vals) > 1 and len(anomaly_vals) > 1:
        normal_mean = np.mean(normal_vals)
        anomaly_mean = np.mean(anomaly_vals)
        n1, n2 = len(normal_vals), len(anomaly_vals)
        s1 = np.std(normal_vals, ddof=1)
        s2 = np.std(anomaly_vals, ddof=1)

        # Weighted pooled standard deviation for imbalanced group sizes
        pooled_var = (((n1 - 1) * s1**2) + ((n2 - 1) * s2**2)) / max(1, (n1 + n2 - 2))
        pooled_std = np.sqrt(max(pooled_var, 0.0))
        effect_size = abs(anomaly_mean - normal_mean) / pooled_std if pooled_std > 0 else 0

        feature_analysis.append({
            'feature': col,
            'normal_mean': normal_mean,
            'anomaly_mean': anomaly_mean,
            'difference': anomaly_mean - normal_mean,
            'effect_size': effect_size,
            'direction': 'higher' if anomaly_mean > normal_mean else 'lower',
            'n_normal': n1,
            'n_anomaly': n2,
        })

feature_analysis = sorted(feature_analysis, key=lambda x: x['effect_size'], reverse=True)

print("Top 10 Most Discriminative Features:")
print("=" * 86)
print(f"{'Feature':<30} {'Effect Size':>12} {'Direction':>10} {'Normal':>10} {'Anomaly':>10} {'n_normal':>10} {'n_anom':>8}")
print("-" * 86)
for item in feature_analysis[:10]:
    print(
        f"{item['feature']:<30} {item['effect_size']:>12.3f} {item['direction']:>10} "
        f"{item['normal_mean']:>10.2f} {item['anomaly_mean']:>10.2f} {item['n_normal']:>10} {item['n_anomaly']:>8}"
    )


In [ ]:
# Visualize top features
top_features = [f['feature'] for f in feature_analysis[:6]]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    feat_idx = feature_cols.index(feat)

    normal_vals = X_valid[normal_mask, feat_idx]
    anomaly_vals = X_valid[anomaly_mask, feat_idx]

    axes[i].hist(normal_vals, bins=30, alpha=0.5, label='Normal', color='green', density=True)
    axes[i].hist(anomaly_vals, bins=30, alpha=0.5, label='Anomaly', color='red', density=True)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'{feat}\n(effect size: {feature_analysis[i]["effect_size"]:.2f})')
    axes[i].legend()

plt.tight_layout()
save_plot(fig, '07_feature_distributions')
plt.show()


## 11. Temporal Analysis (if time column exists)

In [ ]:
if TIME_COL and TIME_COL in df.columns:
    # Create results dataframe for valid samples only
    results_df = df.loc[valid_mask].copy()
    results_df['discovered_label'] = discovered_labels
    results_df['consensus_score'] = consensus_score

    # Parse timestamps
    results_df[TIME_COL] = pd.to_datetime(results_df[TIME_COL], errors='coerce')
    results_df = results_df.dropna(subset=[TIME_COL])

    if len(results_df) > 0:
        # Group by day
        results_df['date'] = results_df[TIME_COL].dt.date

        daily_stats = results_df.groupby('date').agg({
            'consensus_score': ['count', 'mean'],
            'discovered_label': lambda x: x.isin(['high_confidence_anomaly', 'likely_anomaly']).sum()
        }).reset_index()
        daily_stats.columns = ['date', 'total', 'avg_score', 'anomalies']
        daily_stats['anomaly_rate'] = daily_stats['anomalies'] / daily_stats['total']

        # Plot temporal distribution
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))

        # Plot 1: Daily anomaly rate
        axes[0].bar(range(len(daily_stats)), daily_stats['anomaly_rate'], color='coral')
        axes[0].set_ylabel('Anomaly Rate')
        axes[0].set_title('Daily Anomaly Rate')
        axes[0].axhline(y=0.3, color='red', linestyle='--', label='Suspicious threshold (30%)')
        axes[0].legend()

        # Plot 2: Daily sample count
        axes[1].bar(range(len(daily_stats)), daily_stats['total'], color='steelblue', label='Total')
        axes[1].bar(range(len(daily_stats)), daily_stats['anomalies'], color='coral', label='Anomalies')
        axes[1].set_xlabel('Day')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Daily Sample Distribution')
        axes[1].legend()

        plt.tight_layout()
        save_plot(fig, '08_temporal_analysis')
        plt.show()

        # Find suspicious periods
        suspicious = daily_stats[daily_stats['anomaly_rate'] > 0.3]
        if len(suspicious) > 0:
            print("\n⚠️ SUSPICIOUS PERIODS (>30% anomaly rate):")
            for _, row in suspicious.iterrows():
                print(f"   {row['date']}: {row['anomalies']}/{row['total']} anomalies ({row['anomaly_rate']*100:.1f}%)")
        else:
            print("\n✅ No suspicious time periods found.")
else:
    print("No time column available for temporal analysis.")


## 12. Label Quality Visualization & Normal Template

These diagnostic plots help you visually inspect whether the discovered labels align with traffic peaks, calmer periods, and a smooth baseline. The Fourier template source is configurable: you can build it from the robust median of all windows or only from `likely_normal` windows.


In [ ]:
# Build a reusable time-series dataframe for label inspection
LABEL_VIZ_METRIC_CANDIDATES = [
    'total_updates', 'announcements', 'withdrawals',
    'unique_prefixes_ann', 'ann_rate', 'wd_rate'
]
FOURIER_TARGET_CANDIDATES = [
    'total_updates', 'announcements', 'withdrawals', 'unique_prefixes_ann'
]
FOURIER_HARMONICS = 4
ROLLING_WINDOW = 12  # 12 x 5-minute windows ~= 1 hour for the default dataset

LABEL_COLOR_MAP = {
    'likely_normal': '#2ecc71',
    'uncertain': '#f1c40f',
    'likely_anomaly': '#e67e22',
    'high_confidence_anomaly': '#c0392b'
}

label_viz_df = df.loc[valid_mask].copy()
label_viz_df['_source_index'] = label_viz_df.index
label_viz_df['discovered_label'] = discovered_labels
label_viz_df['consensus_score'] = consensus_score
label_viz_df['anomaly_votes'] = anomaly_votes
label_viz_df['is_anomaly'] = label_viz_df['discovered_label'].isin(['high_confidence_anomaly', 'likely_anomaly'])

LABEL_VIZ_METRIC = next((c for c in LABEL_VIZ_METRIC_CANDIDATES if c in label_viz_df.columns), None)
if LABEL_VIZ_METRIC is None:
    LABEL_VIZ_METRIC = feature_analysis[0]['feature'] if feature_analysis else feature_cols[0]

FOURIER_TARGET_METRIC = next((c for c in FOURIER_TARGET_CANDIDATES if c in label_viz_df.columns), LABEL_VIZ_METRIC)

SAMPLE_MINUTES = None
SLOTS_PER_DAY = None

if TIME_COL and TIME_COL in label_viz_df.columns:
    label_viz_df[TIME_COL] = pd.to_datetime(label_viz_df[TIME_COL], errors='coerce')
    label_viz_df = label_viz_df.dropna(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)

    if len(label_viz_df) > 1:
        sample_delta = label_viz_df[TIME_COL].diff().median()
    else:
        sample_delta = pd.Timedelta(minutes=5)

    if pd.isna(sample_delta) or sample_delta <= pd.Timedelta(0):
        sample_delta = pd.Timedelta(minutes=5)

    SAMPLE_MINUTES = max(1, int(round(sample_delta.total_seconds() / 60)))
    SLOTS_PER_DAY = max(1, int(round((24 * 60) / SAMPLE_MINUTES)))

    minutes_of_day = label_viz_df[TIME_COL].dt.hour * 60 + label_viz_df[TIME_COL].dt.minute
    label_viz_df['slot_in_day'] = (minutes_of_day // SAMPLE_MINUTES).astype(int).clip(0, SLOTS_PER_DAY - 1)
    label_viz_df['hour_float'] = minutes_of_day / 60.0

    print(f"Label visualization metric: {LABEL_VIZ_METRIC}")
    print(f"Fourier template metric:   {FOURIER_TARGET_METRIC}")
    print(f"Sampling interval:         {SAMPLE_MINUTES} minutes ({SLOTS_PER_DAY} slots/day)")
    print(f"Time windows available:    {len(label_viz_df)}")
    print(f"Fourier baseline source:   {FOURIER_BASELINE_SOURCE}")
else:
    label_viz_df = pd.DataFrame()
    print("No time column available for label-quality visualization.")


In [ ]:
if label_viz_df.empty:
    print("No label timeline plots were generated because a valid time column was not available.")
else:
    plot_df = label_viz_df.sort_values(TIME_COL).copy()
    metric_series = plot_df[LABEL_VIZ_METRIC].astype(float)
    rolling_baseline = metric_series.rolling(
        ROLLING_WINDOW,
        center=True,
        min_periods=max(3, ROLLING_WINDOW // 3)
    ).median()
    rolling_anomaly_rate = plot_df['is_anomaly'].astype(float).rolling(
        ROLLING_WINDOW,
        min_periods=1
    ).mean()

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                             gridspec_kw={'height_ratios': [2.4, 1.2, 1.2]})

    axes[0].plot(plot_df[TIME_COL], metric_series, color='lightsteelblue', linewidth=0.9,
                 alpha=0.9, label=f'{LABEL_VIZ_METRIC} (observed)')
    axes[0].plot(plot_df[TIME_COL], rolling_baseline, color='navy', linewidth=1.5,
                 label=f'Rolling median ({ROLLING_WINDOW} windows)')

    for label in ['uncertain', 'likely_anomaly', 'high_confidence_anomaly']:
        mask = plot_df['discovered_label'] == label
        if mask.any():
            axes[0].scatter(plot_df.loc[mask, TIME_COL], metric_series[mask],
                            color=LABEL_COLOR_MAP[label], s=18, alpha=0.85, label=label)

    axes[0].set_ylabel(LABEL_VIZ_METRIC)
    axes[0].set_title(f'Label Timeline vs Traffic Behavior ({LABEL_VIZ_METRIC})')
    axes[0].legend(ncol=4, loc='upper right')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(plot_df[TIME_COL], plot_df['consensus_score'], color='#34495e', linewidth=1.1)
    axes[1].axhline(consensus_thresholds['uncertain'], color=LABEL_COLOR_MAP['uncertain'], linestyle='--', linewidth=1,
                    label=f"uncertain threshold ({consensus_thresholds['uncertain']:.3f})")
    axes[1].axhline(consensus_thresholds['likely_anomaly'], color=LABEL_COLOR_MAP['likely_anomaly'], linestyle='--', linewidth=1,
                    label=f"likely_anomaly threshold ({consensus_thresholds['likely_anomaly']:.3f})")
    axes[1].axhline(consensus_thresholds['high_confidence_anomaly'], color=LABEL_COLOR_MAP['high_confidence_anomaly'], linestyle='--', linewidth=1,
                    label=f"high-confidence threshold ({consensus_thresholds['high_confidence_anomaly']:.3f})")
    axes[1].set_ylabel('Consensus score')
    axes[1].set_title('Consensus Score Through Time')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(plot_df[TIME_COL], rolling_anomaly_rate, color='#8e44ad', linewidth=1.4,
                 label=f'Rolling anomaly rate ({ROLLING_WINDOW} windows)')
    axes[2].axhline(0.3, color='crimson', linestyle='--', linewidth=1, label='30% suspicious rate')
    axes[2].set_ylabel('Anomaly rate')
    axes[2].set_xlabel(TIME_COL)
    axes[2].set_title('Local Concentration of Labeled Anomalies')
    axes[2].legend(loc='upper right')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    save_plot(fig, '09_label_quality_timeline')
    plt.show()

    peak_cols = [TIME_COL, LABEL_VIZ_METRIC, 'consensus_score', 'discovered_label']
    peak_windows = plot_df.nlargest(10, LABEL_VIZ_METRIC)[peak_cols].copy()
    peak_windows['consensus_score'] = peak_windows['consensus_score'].round(3)
    print(f"Top 10 peak windows by {LABEL_VIZ_METRIC}:")
    print(peak_windows.to_string(index=False))


In [ ]:
if label_viz_df.empty:
    print("No Fourier template was generated because a valid time column was not available.")
else:
    if FOURIER_BASELINE_SOURCE == 'likely_normal':
        template_source_df = label_viz_df[label_viz_df['discovered_label'] == 'likely_normal'].copy()
        template_source_label = 'likely_normal'
    else:
        template_source_df = label_viz_df.copy()
        template_source_label = 'robust all-data'

    min_points_needed = max(2 * FOURIER_HARMONICS + 1, SLOTS_PER_DAY // 4)
    if FOURIER_BASELINE_SOURCE == 'likely_normal' and len(template_source_df) < min_points_needed:
        print(
            f"Not enough likely_normal windows for a stable Fourier template (need at least {min_points_needed}, found {len(template_source_df)}). Falling back to robust all-data baseline."
        )
        template_source_df = label_viz_df.copy()
        template_source_label = 'robust all-data'

    if len(template_source_df) < min_points_needed:
        print(
            f"Not enough windows to fit a Fourier template from {template_source_label} data (need at least {min_points_needed}, found {len(template_source_df)})."
        )
    else:
        slots = np.arange(SLOTS_PER_DAY)
        theta = 2 * np.pi * slots / SLOTS_PER_DAY

        def fourier_design(theta_values, harmonics):
            columns = [np.ones_like(theta_values)]
            for k in range(1, harmonics + 1):
                columns.append(np.sin(k * theta_values))
                columns.append(np.cos(k * theta_values))
            return np.column_stack(columns)

        slot_group = template_source_df.groupby('slot_in_day')[FOURIER_TARGET_METRIC]
        normal_profile = slot_group.median().reindex(slots)
        profile_q25 = slot_group.quantile(0.25).reindex(slots)
        profile_q75 = slot_group.quantile(0.75).reindex(slots)

        normal_profile = normal_profile.interpolate(limit_direction='both')
        profile_q25 = profile_q25.interpolate(limit_direction='both')
        profile_q75 = profile_q75.interpolate(limit_direction='both')

        design = fourier_design(theta, FOURIER_HARMONICS)
        valid_template_mask = np.isfinite(normal_profile.values)
        coeffs, *_ = np.linalg.lstsq(
            design[valid_template_mask],
            normal_profile.values[valid_template_mask],
            rcond=None
        )
        fourier_profile = design @ coeffs

        label_viz_df['fourier_template'] = fourier_profile[label_viz_df['slot_in_day'].values]
        label_viz_df['fourier_residual'] = label_viz_df[FOURIER_TARGET_METRIC] - label_viz_df['fourier_template']

        residual_reference = label_viz_df.loc[
            label_viz_df['discovered_label'] == 'likely_normal', 'fourier_residual'
        ]
        if residual_reference.empty:
            residual_reference = label_viz_df['fourier_residual']

        residual_scale = residual_reference.std(ddof=0)
        if pd.isna(residual_scale) or residual_scale == 0:
            residual_scale = label_viz_df['fourier_residual'].std(ddof=0)
        if pd.isna(residual_scale) or residual_scale == 0:
            residual_scale = 1.0

        label_viz_df['fourier_residual_z'] = label_viz_df['fourier_residual'] / residual_scale
        slot_hours = slots * SAMPLE_MINUTES / 60.0

        fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False,
                                 gridspec_kw={'height_ratios': [1.4, 1.6]})

        axes[0].plot(slot_hours, normal_profile.values, color=LABEL_COLOR_MAP['likely_normal'],
                     linewidth=2, label=f'{template_source_label} median')
        axes[0].fill_between(slot_hours, profile_q25.values, profile_q75.values,
                             color=LABEL_COLOR_MAP['likely_normal'], alpha=0.2, label=f'{template_source_label} IQR')
        axes[0].plot(slot_hours, fourier_profile, color='navy', linewidth=2.2,
                     label=f'Fourier template ({FOURIER_HARMONICS} harmonics)')
        axes[0].set_ylabel(FOURIER_TARGET_METRIC)
        axes[0].set_xlabel('Hour of day')
        axes[0].set_title(f'Normal Traffic Template by Time of Day ({FOURIER_TARGET_METRIC})')
        axes[0].set_xticks(range(0, 25, 2))
        axes[0].legend(loc='upper right')
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(label_viz_df[TIME_COL], label_viz_df['fourier_residual_z'],
                     color='#6c5ce7', linewidth=1.1, label='Residual z-score')
        axes[1].axhline(3, color='crimson', linestyle='--', linewidth=1, label='+3 sigma')
        axes[1].axhline(-3, color='crimson', linestyle='--', linewidth=1, label='-3 sigma')

        anomaly_mask = label_viz_df['is_anomaly']
        if anomaly_mask.any():
            axes[1].scatter(label_viz_df.loc[anomaly_mask, TIME_COL],
                            label_viz_df.loc[anomaly_mask, 'fourier_residual_z'],
                            color='#c0392b', s=16, alpha=0.85, label='labeled anomaly')

        axes[1].set_xlabel(TIME_COL)
        axes[1].set_ylabel('Residual z-score')
        axes[1].set_title('Deviation from the Fourier Normal Template')
        axes[1].legend(loc='upper right')
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        save_plot(fig, '10_fourier_normal_template')
        plt.show()

        peak_residual_cols = [
            TIME_COL, FOURIER_TARGET_METRIC, 'fourier_template',
            'fourier_residual', 'fourier_residual_z',
            'consensus_score', 'discovered_label'
        ]
        peak_residuals = label_viz_df.nlargest(10, 'fourier_residual')[peak_residual_cols].copy()
        rounded_cols = ['fourier_template', 'fourier_residual', 'fourier_residual_z', 'consensus_score']
        peak_residuals[rounded_cols] = peak_residuals[rounded_cols].round(3)
        print(f"Top 10 positive deviations vs Fourier template ({FOURIER_TARGET_METRIC}, baseline={template_source_label}):")
        print(peak_residuals.to_string(index=False))


## 13. Compare with Existing Labels (if available)

In [ ]:
if EXISTING_LABEL_COL and EXISTING_LABEL_COL in df.columns:
    # Get existing labels for valid samples
    existing_labels = df.loc[valid_mask, EXISTING_LABEL_COL].values
    
    print("Comparison: Existing Labels vs Discovered Labels")
    print("="*70)
    
    comparison_df = pd.DataFrame({
        'existing': existing_labels,
        'discovered': discovered_labels
    })
    
    # Cross-tabulation
    cross_tab = pd.crosstab(comparison_df['existing'], comparison_df['discovered'], 
                            margins=True, normalize='index')
    
    print("\nNormalized by existing label (row-wise):")
    print(cross_tab.round(3))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    cross_tab_counts = pd.crosstab(comparison_df['existing'], comparison_df['discovered'])
    sns.heatmap(cross_tab_counts, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    plt.title('Existing Labels vs Discovered Labels')
    plt.xlabel('Discovered Label')
    plt.ylabel('Existing Label')
    plt.tight_layout()
    save_plot(fig, '11_label_comparison')
    plt.show()
    
    # Recommendations
    print("\n📋 RECOMMENDATIONS:")
    for existing_label in df[EXISTING_LABEL_COL].unique():
        mask = existing_labels == existing_label
        discovered_dist = pd.Series(discovered_labels[mask]).value_counts(normalize=True)
        
        anomaly_rate = discovered_dist.get('high_confidence_anomaly', 0) + discovered_dist.get('likely_anomaly', 0)
        normal_rate = discovered_dist.get('likely_normal', 0)
        
        if 'normal' in str(existing_label).lower() and anomaly_rate > 0.2:
            print(f"   ⚠️ '{existing_label}': {anomaly_rate*100:.1f}% look anomalous - review these samples!")
        elif 'attack' in str(existing_label).lower() or 'hijack' in str(existing_label).lower():
            if normal_rate > 0.3:
                print(f"   ⚠️ '{existing_label}': {normal_rate*100:.1f}% look normal - possible mislabeling!")
            else:
                print(f"   ✅ '{existing_label}': Looks correctly labeled ({anomaly_rate*100:.1f}% detected as anomalous)")
        else:
            print(f"   ℹ️ '{existing_label}': {normal_rate*100:.1f}% normal, {anomaly_rate*100:.1f}% anomalous")
else:
    print("No existing labels to compare with.")

## 14. Save Results

In [ ]:
# Create final results dataframe
results_df = df.copy()

# Add results for valid samples
results_df['iso_forest_score'] = np.nan
results_df['lof_score'] = np.nan
results_df['statistical_score'] = np.nan
results_df['elliptic_score'] = np.nan
results_df['hdbscan_outlier_score'] = np.nan
results_df['hdbscan_unscored'] = False
results_df['equal_consensus_score'] = np.nan
results_df['fourier_template'] = np.nan
results_df['fourier_residual'] = np.nan
results_df['fourier_residual_z'] = np.nan
results_df['cluster'] = -1
results_df['hdbscan_cluster'] = -1
results_df['kmeans_cluster'] = -1
results_df['kmeans_small_cluster_flag'] = False
results_df['kmeans_label_boosted'] = False
results_df['anomaly_votes'] = np.nan
results_df['consensus_score'] = np.nan
results_df['discovered_label_pre_kmeans'] = 'invalid'
results_df['discovered_label'] = 'invalid'
results_df['label_stability_score'] = np.nan
results_df['label_flip_rate'] = np.nan
results_df['bootstrap_modal_label'] = ''
results_df['bootstrap_modal_agreement'] = np.nan

results_df.loc[valid_mask, 'iso_forest_score'] = iso_scores
results_df.loc[valid_mask, 'lof_score'] = lof_scores
results_df.loc[valid_mask, 'statistical_score'] = stat_scores
results_df.loc[valid_mask, 'elliptic_score'] = elliptic_scores
results_df.loc[valid_mask, 'hdbscan_outlier_score'] = hdbscan_outlier_scores
results_df.loc[valid_mask, 'hdbscan_unscored'] = hdbscan_unscored_mask
results_df.loc[valid_mask, 'equal_consensus_score'] = equal_consensus_score
results_df.loc[valid_mask, 'cluster'] = hdbscan_labels
results_df.loc[valid_mask, 'hdbscan_cluster'] = hdbscan_labels
results_df.loc[valid_mask, 'kmeans_cluster'] = cluster_labels
results_df.loc[valid_mask, 'kmeans_small_cluster_flag'] = kmeans_small_cluster_flag
results_df.loc[valid_mask, 'kmeans_label_boosted'] = kmeans_label_boosted
results_df.loc[valid_mask, 'anomaly_votes'] = anomaly_votes
results_df.loc[valid_mask, 'consensus_score'] = consensus_score
results_df.loc[valid_mask, 'discovered_label_pre_kmeans'] = discovered_labels_pre_kmeans
results_df.loc[valid_mask, 'discovered_label'] = discovered_labels

if 'label_stability_score' in globals():
    results_df.loc[valid_mask, 'label_stability_score'] = label_stability_score
    results_df.loc[valid_mask, 'label_flip_rate'] = label_flip_rate
    results_df.loc[valid_mask, 'bootstrap_modal_label'] = bootstrap_modal_label
    results_df.loc[valid_mask, 'bootstrap_modal_agreement'] = bootstrap_modal_agreement

if 'label_viz_df' in globals() and not label_viz_df.empty and '_source_index' in label_viz_df.columns:
    fourier_cols = [c for c in ['fourier_template', 'fourier_residual', 'fourier_residual_z'] if c in label_viz_df.columns]
    for col in fourier_cols:
        results_df.loc[label_viz_df['_source_index'], col] = label_viz_df[col].values

# Save to file
input_path = Path(INPUT_FILE)
output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else input_path.parent
output_path = output_dir / f"{input_path.stem}_discovered.csv"

results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

# Save anomalies separately
anomalies_df = results_df[results_df['discovered_label'].isin(['high_confidence_anomaly', 'likely_anomaly'])]
if len(anomalies_df) > 0:
    anomalies_path = output_dir / f"{input_path.stem}_anomalies.csv"
    anomalies_df.to_csv(anomalies_path, index=False)
    print(f"Anomalies saved to: {anomalies_path} ({len(anomalies_df)} samples)")

# Save metadata sidecar for reproducibility
import json as jsonlib

def to_python(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [to_python(v) for v in value.tolist()]
    if isinstance(value, pd.Series):
        return {str(k): to_python(v) for k, v in value.items()}
    if isinstance(value, pd.DataFrame):
        return [{str(k): to_python(v) for k, v in row.items()} for row in value.to_dict(orient='records')]
    if isinstance(value, dict):
        return {str(k): to_python(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_python(v) for v in value]
    return value

metadata = {
    'input_file': str(input_path),
    'output_csv': str(output_path),
    'anomaly_rate_mode': ANOMALY_RATE_MODE,
    'fixed_contamination_reference': float(FIXED_CONTAMINATION),
    'consensus_mode': consensus_mode,
    'method_weights': to_python(method_weights),
    'consensus_thresholds': to_python(consensus_thresholds),
    'active_methods': list(active_methods),
    'inactive_methods': list(inactive_methods),
    'effective_method_count': int(effective_method_count),
    'lof_n_neighbors': int(N_NEIGHBORS_LOF),
    'hdbscan': {
        'min_cluster_size': int(MIN_CLUSTER_SIZE_HDBSCAN),
        'min_samples': int(MIN_SAMPLES_HDBSCAN),
        'knee_search_fraction': float(HDBSCAN_KNEE_SEARCH_FRAC),
        'top_gaps_reported': int(HDBSCAN_TOP_GAPS_TO_REPORT),
        'fallback_threshold': float(HDBSCAN_OUTLIER_THRESHOLD_FALLBACK),
        'threshold_used': float(outlier_threshold),
        'threshold_description': threshold_desc,
        'unscored_as_suspicious': bool(HDBSCAN_UNSCORED_AS_SUSPICIOUS),
        'unscored_count': int(hdbscan_unscored_mask.sum()),
        'top_gaps': to_python(knee_result['top_gaps']) if knee_result is not None else [],
    },
    'kmeans': {
        'small_cluster_fraction': float(KMEANS_SMALL_CLUSTER_FRAC),
        'upgrade_uncertain': bool(KMEANS_UPGRADE_UNCERTAIN),
        'small_clusters': [int(x) for x in small_cluster_ids],
        'upgraded_windows': int(kmeans_label_boosted.sum()),
    },
    'bootstrap_stability': {
        'iterations': int(BOOTSTRAP_STABILITY_ITERATIONS),
        'sample_fraction': float(BOOTSTRAP_SAMPLE_FRAC),
        'sampling_strategy': 'subsample_without_replacement',
        'sampling_note': 'Subsampling stability test for label robustness, not a classical bootstrap confidence interval.',
        'mean_stability': float(np.mean(label_stability_score)) if 'label_stability_score' in globals() else None,
        'median_stability': float(np.median(label_stability_score)) if 'label_stability_score' in globals() else None,
    },
    'leave_one_method_out': to_python(leave_one_method_out_df) if 'leave_one_method_out_df' in globals() else [],
    'leave_one_method_out_by_method': {
        str(row['dropped_method']): {k: to_python(v) for k, v in row.items() if k != 'dropped_method'}
        for row in leave_one_method_out_df.to_dict(orient='records')
    } if 'leave_one_method_out_df' in globals() else {},
    'fourier': {
        'baseline_source': FOURIER_BASELINE_SOURCE,
    },
}

metadata_path = output_dir / f"{input_path.stem}_labeling_metadata.json"
with metadata_path.open('w', encoding='utf-8') as f:
    jsonlib.dump(to_python(metadata), f, indent=2)
print(f"Metadata saved to: {metadata_path}")
